In [8]:
import pandas as pd
import numpy as np
import unicodedata, re
from collections import Counter

# load raw
df = pd.read_csv("routes.csv")
df.head(5)

,log_id,logged_on,office,from_town,to_town,route_type
0,LOG-00000,2024-01-01,OFF-07,Imatra.,Ingå,bus
1,LOG-00001,2024-01-01,OFF-06,Harjavalta,Pirkkala,ferry
2,LOG-00002,2024-01-01,OFF-09,Imatra,Lieksa,ferry
3,LOG-00003,2024-01-01,OFF-10,Kankaanpää,Helsingfors,bus
4,LOG-00004,2024-01-01,OFF-09,Villmanstrand,helsingfors,ferry


In [9]:
# Mapping dictionary to canon
SWEDISH_TO_FINNISH = {
    'abo': 'Turku', 'tammerfors': 'Tampere', 'helsingfors': 'Helsinki',
    'bjorneborg': 'Pori', 'borga': 'Porvoo', 'brahestad': 'Raahe',
    'esbo': 'Espoo', 'fredrikshamn': 'Hamina', 'grankulla': 'Kauniainen',
    'hango': 'Hanko', 'hyvinge': 'Hyvinkää', 'idensalmi': 'Iisalmi',
    'ikalis': 'Ikaalinen', 'ilmola': 'Ilmajoki', 'inga': 'Inkoo',
    'jakobstad': 'Pietarsaari', 'jarvenpaa': 'Järvenpää', 'kajana': 'Kajaani',
    'karleby': 'Kokkola', 'kaskiko': 'Kaskinen', 'kasko': 'Kaskinen',
    'kemionsaari': 'Kemiönsaari', 'kimitoon': 'Kemiönsaari', 'kino': 'Kemiönsaari',
    'kumo': 'Kokemäki', 'kyrkslatt': 'Kirkkonummi', 'lahtis': 'Lahti',
    'lovisa': 'Loviisa', 'lundo': 'Lieto', 'malax': 'Maalahti',
    'mustasaari': 'Vaasa', 'nadendal': 'Naantali', 'narpes': 'Närpiö',
    'narpio': 'Närpiö', 'nykarleby': 'Uusikaarlepyy', 'nystad': 'Uusikaupunki',
    'pargas': 'Parainen', 'raumo': 'Rauma', 'raseborg': 'Raasepori',
    'reso': 'Raisio', 'st karins': 'Kaarina', 'st michel': 'Mikkeli',
    'tavastehus': 'Hämeenlinna', 'tornea': 'Tornio', 'traskanda': 'Järvenpää',
    'uleaborg': 'Oulu', 'ulvsby': 'Ulvila', 'vanda': 'Vantaa',
    'vasa': 'Vaasa', 'vichtis': 'Vihti', 'villmanstrand': 'Lappeenranta',
    'virmo': 'Mynämäki', 'vora': 'Vöyri', 'voyri': 'Vöyri'
}

def normalize_name(s):
    if not isinstance(s, str):
        return ""
    s = s.strip().rstrip(".,")
    s = unicodedata.normalize("NFC", s)
    # strip for dictionary
    key = "".join(c for c in unicodedata.normalize("NFD", s.lower()) if unicodedata.category(c) != "Mn")
    key = re.sub(r"\s+", " ", key).strip()

    if key in SWEDISH_TO_FINNISH:
        return SWEDISH_TO_FINNISH[key]

    return s.capitalize()

In [10]:
# town columns cleaning
df["from_clean"] = df["from_town"].apply(normalize_name)
df["to_clean"] = df["to_town"].apply(normalize_name)

# filter out self-loops
initial_rows = len(df)
df = df[df["from_clean"] != df["to_clean"]].copy()
print(f"Filtered out {initial_rows - len(df)} self-loops. Rows remaining: {len(df)}")

Filtered out 0 self-loops. Rows remaining: 838


In [11]:
# Create canonical undirected edge pairs
df["pair"] = [tuple(sorted((a, b))) for a, b in zip(df["from_clean"], df["to_clean"])]
unique_edges = set(df["pair"])

unique_towns = {t for edge in unique_edges for t in edge}
print(f"Cleaned Network Stats -> Towns: {len(unique_towns)} | Unique Connections: {len(unique_edges)}")

Cleaned Network Stats -> Towns: 152 | Unique Connections: 499


In [12]:
# Calculate degree counts for each town
deg = Counter(town for edge in unique_edges for town in edge)
top7_towns = [t for t, _ in deg.most_common(7)]

print("Top 7 Busiest Towns:")
for idx, (town, count) in enumerate(deg.most_common(7), 1):
    print(f"  {idx}. {town}: {count} connections")

# decapitating and recapitating
word = "".join(t[0].upper() for t in top7_towns)
flag = f"SIGMOID_{word}"

print(f"\n FINAL FLAG: {flag}")

Top 7 Busiest Towns:
  1. Orivesi: 44 connections
  2. Helsinki: 41 connections
  3. Tornio: 33 connections
  4. Lieksa: 31 connections
  5. Imatra: 30 connections
  6. Nokia: 25 connections
  7. Närpiö: 25 connections

 FINAL FLAG: SIGMOID_OHTLINN
